# Milestone 0.4: Python Optimization - Code Profiling

This notebook introduces **code profiling**, the process of analyzing program execution to identify performance bottlenecks – parts of the code that consume the most time or resources.

**Goal:** Learn how to use Python's built-in `cProfile` module to measure the performance of different parts of a Python script.

**Reference Script:** `profiling_example.py`

## Why Profile Code?

Before optimizing code, it's crucial to know *where* the performance issues lie. Profiling helps you:

1.  **Identify Bottlenecks:** Find the specific functions or lines of code that take the longest to execute.
2.  **Focus Optimization Efforts:** Avoid wasting time optimizing code that isn't actually slow (premature optimization).
3.  **Understand Code Behavior:** Gain insights into how often functions are called and how much time is spent within them versus functions they call.

Python provides several profiling tools. We'll focus on `cProfile`, which is built-in and suitable for most use cases. It measures the time spent in function calls.

## Using `cProfile` and `pstats`

The typical workflow involves:
1.  Importing `cProfile` and `pstats`.
2.  Creating a `cProfile.Profile()` object.
3.  Enabling the profiler (`profiler.enable()`).
4.  Running the code you want to profile.
5.  Disabling the profiler (`profiler.disable()`).
6.  Using `pstats.Stats()` to load the profiling data.
7.  Sorting and printing the statistics (`stats.sort_stats().print_stats()`).

In [1]:
import cProfile
import pstats
import io # To capture stats output in the notebook
import time

# Import the function to profile from our script
try:
    # Assuming the notebook and script are in the same directory
    # Or that the 'phase0.python_optimization' package is importable
    from phase0.python_optimization.profiling_example import loop_addition 
except ImportError:
     print("Error: Could not import from profiling_example.py. Make sure it's in the correct path.")
     # Define it here if import fails, for demonstration purposes
     def loop_addition(list1, list2):
         if len(list1) != len(list2):
             raise ValueError("Lists must have the same length")
         result = [0] * len(list1)
         for i in range(len(list1)):
             result[i] = list1[i] + list2[i]
         return result

Starting profiling for data size: 1000000...
Profiling finished.

--- Profiling Results (Top 10 by cumulative time) ---
         6 function calls in 0.081 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.081    0.081    0.081    0.081 /home/jakedevsthings/ai/learn-ai-ml-dl/phase0/python_optimization/profiling_example.py:9(loop_addition)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        4    0.000    0.000    0.000    0.000 {built-in method builtins.len}





In [2]:
# Prepare data for the function
data_size = 1_000_000 # Same size as in the script
list_a = list(range(data_size))
list_b = list(range(data_size))

print(f"Prepared data with size: {data_size:,}")

Prepared data with size: 1,000,000


### Running the Profiler

In [3]:
# 1. Create a Profile object
profiler = cProfile.Profile()

# 2. Enable profiling
print("Starting profiling...")
profiler.enable()

# 3. Run the code to be profiled
result = loop_addition(list_a, list_b)

# 4. Disable profiling
profiler.disable()
print("Profiling finished.")

Starting profiling...
Profiling finished.


### Analyzing the Results

Now, we use `pstats` to format and display the collected data. We'll sort the results by `cumulative` time, which is the total time spent in a function, including time spent in sub-functions it calls. This often helps identify the high-level bottlenecks.

In [4]:
# 5. Create a stream to capture the output
stream = io.StringIO()

# 6. Load stats into pstats, sorting by cumulative time
# Other sort keys: 'calls', 'time' (total time excluding sub-functions), 'filename', 'name'
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumulative')

# 7. Print the stats (e.g., top 10 lines)
print("Profiling Results (Sorted by Cumulative Time) ---")
stats.print_stats(10) # Show top 10 entries

# Print the captured output
print(stream.getvalue())

Profiling Results (Sorted by Cumulative Time) ---
         51 function calls in 0.083 seconds

   Ordered by: cumulative time
   List reduced from 22 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.083    0.042 /home/jakedevsthings/ai/learn-ai-ml-dl/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3513(run_code)
        2    0.000    0.000    0.083    0.042 {built-in method builtins.exec}
        1    0.000    0.000    0.083    0.083 /tmp/ipykernel_21407/3311616692.py:1(<module>)
        1    0.083    0.083    0.083    0.083 /home/jakedevsthings/ai/learn-ai-ml-dl/phase0/python_optimization/profiling_example.py:9(loop_addition)
        2    0.000    0.000    0.000    0.000 /usr/lib/python3.11/codeop.py:120(__call__)
        2    0.000    0.000    0.000    0.000 {built-in method builtins.compile}
        2    0.000    0.000    0.000    0.000 /home/jakedevsthings/ai/learn-ai-ml-dl/ven

#### Understanding the Output

The output shows several columns:

*   `ncalls`: Number of times the function was called.
*   `tottime`: Total time spent *within* this function (excluding time in sub-functions).
*   `percall`: `tottime` divided by `ncalls`.
*   `cumtime`: Cumulative time spent in this function *and* all sub-functions called from it.
*   `percall`: `cumtime` divided by `ncalls`.
*   `filename:lineno(function)`: The function's location.

Look for functions with high `cumtime` and `tottime`. In this example, you should see that the `loop_addition` function itself consumes most of the time, as expected.

### Sorting by Other Criteria

Sometimes it's useful to sort by `tottime` (also called `internal time` or `time`) to see which functions spent the most time executing their own code, excluding calls to other functions.

In [5]:
# Reset the stream and print sorted by 'tottime'
stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('time') # 'time' is equivalent to 'tottime'

print("Profiling Results (Sorted by Total Time / Tottime) ---")
stats.print_stats(10) 
print(stream.getvalue())

Profiling Results (Sorted by Total Time / Tottime) ---
         51 function calls in 0.083 seconds

   Ordered by: internal time
   List reduced from 22 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.083    0.083    0.083    0.083 /home/jakedevsthings/ai/learn-ai-ml-dl/phase0/python_optimization/profiling_example.py:9(loop_addition)
        2    0.000    0.000    0.000    0.000 {built-in method builtins.compile}
        2    0.000    0.000    0.000    0.000 /usr/lib/python3.11/codeop.py:120(__call__)
        2    0.000    0.000    0.083    0.042 /home/jakedevsthings/ai/learn-ai-ml-dl/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3513(run_code)
        2    0.000    0.000    0.000    0.000 /home/jakedevsthings/ai/learn-ai-ml-dl/venv/lib/python3.11/site-packages/traitlets/traitlets.py:676(__get__)
        2    0.000    0.000    0.000    0.000 /usr/lib/python3.11/contextlib.py:104(__init__)
       

## Saving and Visualizing Profile Data (Optional)

For complex codebases, visualizing profile data can be very helpful. You can save the `cProfile` output to a file and use tools like `snakeviz`.

```python
# Save stats to a file
profile_filename = 'loop_addition_profile.prof'
profiler.dump_stats(profile_filename)
print(f"
Profile data saved to {profile_filename}")
```

Then, from your terminal (after installing snakeviz: `pip install snakeviz`), you can run:

```bash
snakeviz loop_addition_profile.prof
```

This will open an interactive visualization in your web browser.

In [6]:
# Example of saving the profile data
profile_filename = 'loop_addition_profile.prof'
profiler.dump_stats(profile_filename)
print(f"Profile data saved to {profile_filename}")
print("You can now analyze this file using: pip install snakeviz && snakeviz loop_addition_profile.prof")

Profile data saved to loop_addition_profile.prof
You can now analyze this file using: pip install snakeviz && snakeviz loop_addition_profile.prof


## Conclusion

Profiling is an essential first step in optimization. `cProfile` provides a powerful way to understand where your Python code spends its time. By identifying bottlenecks, you can focus your optimization efforts effectively, such as applying vectorization (as seen in the previous notebook) where it matters most.